In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from calendar import monthrange
import uuid
import time

class EEXScraper:
    """Scraper EEX - Prix électricité (Base)"""

    API_URL = "https://api.eex-group.com/pub/market-data/table-data"

    HEADERS = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Referer": "https://www.eex.com/",
        "Origin": "https://www.eex.com"
    }

    SHORT_CODES = {
        "Month": "F7BM",
        "Quarter": "F7BQ",
        "Year": "F7BY"
    }

    TYPE_MAPPING = {
        "Month": "mensuel",
        "Quarter": "trimestriel",
        "Year": "annuel"
    }

    MONTHS = {
        "january": 1, "february": 2, "march": 3, "april": 4,
        "may": 5, "june": 6, "july": 7, "august": 8,
        "september": 9, "october": 10, "november": 11, "december": 12
    }

    MONTHS_FR = {
        "January": "Janvier", "February": "Fevrier", "March": "Mars",
        "April": "Avril", "May": "Mai", "June": "Juin",
        "July": "Juillet", "August": "Aout",
        "September": "Septembre", "October": "Octobre",
        "November": "Novembre", "December": "Decembre"
    }

    MONTHS_EN = [
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"
    ]

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(self.HEADERS)

    # -------------------- API --------------------
    def fetch_market_data(self, maturity_type, delivery, product="Base", area="FR",
                          start_date=None, end_date=None):
        short_code, maturity = self.parse_delivery(maturity_type, delivery)

        if not end_date:
            end_date = datetime.now().strftime("%Y-%m-%d")
        if not start_date:
            start_date = (datetime.now() - timedelta(days=5)).strftime("%Y-%m-%d")

        params = {
            "shortCode": short_code,
            "commodity": "POWER",
            "pricing": "F",
            "area": area,
            "product": product,
            "maturity": maturity,
            "maturityType": maturity_type,
            "isRolling": "true",
            "startDate": start_date,
            "endDate": end_date
        }

        try:
            response = self.session.get(self.API_URL, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            return None

    def get_latest_price(self, response):
        if not response or not response.get("data"):
            return None
        try:
            headers = response["header"]
            idx_date = headers.index("tradeDate")
            idx_price = headers.index("settlPx")
            latest_row = max(response["data"], key=lambda r: datetime.fromisoformat(r[idx_date]))
            return latest_row[idx_price]
        except Exception:
            return None

    # -------------------- Parsing --------------------
    def parse_delivery(self, maturity_type, delivery):
        short_code = self.SHORT_CODES[maturity_type]
        if maturity_type == "Month":
            m, y = delivery.split()
            maturity = f"{y}{self.MONTHS[m.lower()]:02d}"
        elif maturity_type == "Quarter":
            q = int(delivery[1])
            y = int(delivery.split()[-1])
            month = {1: 1, 2: 4, 3: 7, 4: 10}[q]
            maturity = f"{y}{month:02d}"
        else:  # Year
            year = int("20" + delivery.split("_")[1])
            maturity = f"{year}01"
        return short_code, maturity

    def compute_period_dates(self, maturity_type, delivery):
        if maturity_type == "Month":
            m, y = delivery.split()
            month = self.MONTHS[m.lower()]
            year = int(y)
            start = datetime(year, month, 1)
            last_day = monthrange(year, month)[1]
            end = datetime(year, month, last_day)
        elif maturity_type == "Quarter":
            q = int(delivery[1])
            year = int(delivery.split()[-1])
            start_month, end_month = {1: (1, 3), 2: (4, 6), 3: (7, 9), 4: (10, 12)}[q]
            start = datetime(year, start_month, 1)
            last_day = monthrange(year, end_month)[1]
            end = datetime(year, end_month, last_day)
        else:
            year = int("20" + delivery.split("_")[1])
            start = datetime(year, 1, 1)
            end = datetime(year, 12, 31)
        return start, end

    def format_periode_label(self, maturity_type, delivery):
        if maturity_type == "Month":
            for en, fr in self.MONTHS_FR.items():
                delivery = delivery.replace(en, fr)
            return delivery
        if maturity_type == "Quarter":
            q = delivery[1]
            y = delivery.split()[-1]
            return { "1": f"1er trimestre {y}", "2": f"2eme trimestre {y}",
                     "3": f"3eme trimestre {y}", "4": f"4eme trimestre {y}" }[q]
        year = "20" + delivery.split("_")[1]
        return f"Annee {year}"

    # -------------------- Construction --------------------
    def build_price_row(self, maturity_type, delivery, area="FR"):
        try:
            base = self.fetch_market_data(maturity_type, delivery, "Base", area)
            base_px = self.get_latest_price(base)
        except Exception:
            base_px = None

        date_deb, date_fin = self.compute_period_dates(maturity_type, delivery)

        return {
            "periode": self.format_periode_label(maturity_type, delivery),
            "prix_base": base_px,
            "type": self.TYPE_MAPPING[maturity_type],
            "date_maj": datetime.now(),
            "date_deb": date_deb,
            "date_fin": date_fin,
            "id_prev_prix": str(uuid.uuid4())
        }

    def build_prices_table(self, configs, area="FR"):
        rows = []
        for maturity_type, delivery in configs:
            row = self.build_price_row(maturity_type, delivery, area)
            print(f"Récupération: {maturity_type} {delivery} → prix_base={row['prix_base']}")
            rows.append(row)
            time.sleep(0.5)  # anti rate-limit soft
        return pd.DataFrame(rows)

# -------------------- Génération des périodes --------------------
def generate_next_3_years_periods():
    today = datetime.now()
    current_year = today.year
    configs = []

    MONTHS_EN = [
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"
    ]

    for year in range(current_year, current_year + 3):
        # Mois
        for month in range(1, 13):
            month_name = MONTHS_EN[month - 1]
            configs.append(("Month", f"{month_name} {year}"))

        # Quarters
        for q in range(1, 5):
            configs.append(("Quarter", f"Q{q} {year}"))
        # Année
        cal = f"Cal_{str(year)[2:]}"  # ex: 2026 → Cal_26
        configs.append(("Year", cal))
    return configs

# -------------------- EXÉCUTION --------------------
if __name__ == "__main__":
    scraper = EEXScraper()
    configs = generate_next_3_years_periods()
    df = scraper.build_prices_table(configs)
    print(df.head(15))
    df.to_csv("prix_energie_3_ans.csv", index=False)


NameError: name 'self' is not defined